# 🦀 Day 5: Core Data Models and Traits — Implement Foundation Types
---

Today we implement the core data types for RustKV.

- **Value enum**: String, Integer, Float, List, Hash
- **Entry struct**: value, created_at, expires_at
- **Command enum**: Get, Set, Delete, Keys, Ping, Quit
- **Response enum**: Value, Ok, Error, Nil, Array
- **Derive**: Display, Debug, Clone, PartialEq; serde for serialization
- **Tests**: Comprehensive tests for all types

In [ ]:
:dep serde = { version = "1", features = ["derive"] }
:dep serde_json = "1"

In [ ]:
use std::collections::HashMap;
use std::fmt;
use std::time::{Duration, Instant};

/// Supported value types for RustKV
#[derive(Debug, Clone, PartialEq, serde::Serialize, serde::Deserialize)]
pub enum Value {
    String(String),
    Integer(i64),
    Float(f64),
    List(Vec<String>),
    Hash(HashMap<String, String>),
}

impl fmt::Display for Value {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        match self {
            Value::String(s) => write!(f, "{}", s),
            Value::Integer(i) => write!(f, "{}", i),
            Value::Float(x) => write!(f, "{}", x),
            Value::List(v) => write!(f, "{:?}", v),
            Value::Hash(h) => write!(f, "{:?}", h),
        }
    }
}

/// Stored entry with metadata
#[derive(Debug, Clone)]
pub struct Entry {
    pub value: Value,
    pub created_at: Instant,
    pub expires_at: Option<Instant>,
}

impl Entry {
    pub fn new(value: Value, ttl: Option<Duration>) -> Self {
        let created_at = Instant::now();
        let expires_at = ttl.map(|d| created_at + d);
        Self { value, created_at, expires_at }
    }

    pub fn is_expired(&self) -> bool {
        self.expires_at.map_or(false, |t| Instant::now() >= t)
    }
}

let entry = Entry::new(Value::String("hello".into()), Some(Duration::from_secs(60)));
println!("Entry: {:?}, expired: {}", entry.value, entry.is_expired());

In [ ]:
/// Commands the client can send
#[derive(Debug, Clone, PartialEq)]
pub enum Command {
    Get(String),
    Set(String, Value, Option<Duration>),
    Delete(String),
    Keys(String),
    Ping,
    Quit,
}

/// Server responses
#[derive(Debug, Clone, PartialEq)]
pub enum Response {
    Value(Value),
    Ok,
    Error(String),
    Nil,
    Array(Vec<Response>),
}

// Example usage
let cmd = Command::Set("name".into(), Value::String("RustKV".into()), Some(Duration::from_secs(30)));
let resp = Response::Ok;
println!("Command: {:?}", cmd);
println!("Response: {:?}", resp);

In [ ]:
// Serialization with serde
let v = Value::Hash([("name".into(), "RustKV".into())].into_iter().collect());
let json = serde_json::to_string(&v).unwrap();
println!("Serialized: {}", json);
let back: Value = serde_json::from_str(&json).unwrap();
assert_eq!(v, back);

In [ ]:
// Tests for core types (EvCxR: use #[test] in a crate; here we demonstrate)

fn test_entry_expiration() {
    use std::time::Duration;
    let entry = Entry::new(Value::Integer(42), Some(Duration::from_secs(0)));
    std::thread::sleep(Duration::from_millis(10));
    assert!(entry.is_expired(), "Entry with 0s TTL should be expired");
}

fn test_entry_no_ttl() {
    let entry = Entry::new(Value::String("permanent".into()), None);
    assert!(!entry.is_expired(), "Entry without TTL should not expire");
}

test_entry_no_ttl();
// test_entry_expiration(); // May be flaky in notebook
println!("Tests passed!");

## 📝 Exercise: Add a New Value Type

Add a new variant to `Value`: either `Set(std::collections::HashSet<String>)` or `SortedSet(Vec<(String, f64)>)`.

1. Add the variant
2. Update `Display` impl
3. Add a `Command` variant if needed (e.g., `SAdd`, `ZAdd`)
4. Add a test

In [ ]:
// YOUR CODE HERE
// Add Set or SortedSet to Value enum

use std::collections::HashSet;

// Uncomment and extend:
// #[derive(Debug, Clone, PartialEq)]
// pub enum Value {
//     ...
//     Set(HashSet<String>),
// }

let my_set: HashSet<String> = ["a".into(), "b".into(), "c".into()].into();
println!("Set: {:?}", my_set);

## 🎯 Key Takeaways

1. **Value** enum: String, Integer, Float, List, Hash — extensible via new variants
2. **Entry** struct: value + created_at + expires_at; `is_expired()` for TTL
3. **Command** enum: Get, Set, Delete, Keys, Ping, Quit
4. **Response** enum: Value, Ok, Error, Nil, Array
5. Implement `Display`, `Debug`, `Clone`, `PartialEq` for all types
6. Use serde for serialization when persistence is needed

---
**Tomorrow:** Core logic part 1 — in-memory storage engine